# 综合实训 · 存储层次实测与访存优化

**所属**：《并行计算》第二章 · 并行软硬件架构　|　**难度**：⭐⭐⭐ 综合　|　**预计时长**：40-60 分钟

> **实验说明**
> 1. 本实验是第二章的**综合实训**。第二章讲授的内容是并行软硬件架构，是本课程的基础，其中存储系统是性能的核心关键之一。本实验将实测计算机存储系统的定量结果。
> 2. 实验由三个递进的对照实验构成，分别回答同一问题的三个层面：
>    **遍历顺序实验**——计算量相同、仅循环顺序不同时，执行时间为何相差一个数量级；
>    **访存步长实验**——一次内存读取的实际代价是多少，为何读取次数减少十六倍并不能节省相应的时间；
>    **指针追逐实验**——一次读取需要等待多久，该数值如何取决于数据所处的存储层级。
> 3. 三个实验相互印证：遍历顺序实验提出现象，访存步长实验解释机制并测定缓存行大小，指针追逐实验给出完整的层次结构并测定各级缓存容量。最后使用测得的硬件参数，定量解释遍历顺序实验中的每一个比值。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 本实验仅依赖标准 C 库，对处理器架构无特殊要求；但**必须在真实硬件上运行**。在指令级模拟器（如 QEMU 用户态）中，缓存层次被完全绕过，所测得的时间数据不具备物理意义。


## 🎯 学习目标

完成本任务的学习后，学生应能够：

- 说明操作次数相同为何不意味着执行时间相同，并指出决定执行时间的第二个因素是**访存模式**
- 描述**存储层次**的结构，说出寄存器、L1、L2、L3 与主存在容量与延迟上的数量级差异
- 解释**缓存行**（cache line）的概念，说明内存搬运的最小单位为何是一整条缓存行而非单个变量
- 区分**空间局部性**与**时间局部性**，并判断给定代码利用的是其中哪一种
- 说明**硬件预取器**的作用范围与能力边界：它能够隐藏访存延迟，但不能减少数据搬运量
- 说明**指针追逐**（pointer chasing）方法的原理，解释它为何能够同时使乱序执行与硬件预取失效
- 从实测曲线的拐点反推本机的**缓存行大小**与**各级缓存容量**，并与内核发布的硬件参数相互核对
- 将上述认识转化为具体的优化手段：**分块**（blocking / tiling）


## 🗺️ 学习路径

1. **提出问题**：同一份数据、相同次数的加法运算，仅交换两层循环的次序，执行时间即相差一个数量级
2. **遍历顺序实验**：以行优先与列优先两种次序遍历同一矩阵，规模自可完全驻留缓存逐步增大至远超缓存容量
   → 现象：矩阵规模越大，两者差距越显著
3. **访存步长实验**：固定一个远大于末级缓存的数组，仅改变访问步长，观察读取次数减半而执行时间并未相应减半的现象
   → 机制：内存以**缓存行**为单位搬运；同时测定本机的有效搬运粒度，并观察硬件预取器的作用
4. **指针追逐实验**：以随机化的指针追逐测定不同工作集规模下的单次访问延迟
   → 全景：曲线上的每一级台阶对应一级缓存的容量边界
5. **相互印证**：使用后两个实验测得的硬件参数，逐项解释遍历顺序实验表格中的比值
6. **🚀 分块转置实验**：矩阵转置中列优先访问无法回避，采用**分块**方法予以优化


## 1. 问题的提出

以下两段代码将同一个 $N \times N$ 矩阵的全部元素求和。二者读取的元素完全相同，加法运算的次数完全相同，所生成指令的种类也基本一致：

```c
/* 版本 A：行优先 */                   /* 版本 B：列优先 */
for (i = 0; i < N; i++)               for (j = 0; j < N; j++)
  for (j = 0; j < N; j++)               for (i = 0; i < N; i++)
    sum += a[i * N + j];                  sum += a[i * N + j];
```

二者唯一的差别是两层循环的次序。请在运行程序之前先做出定量预测，并将预测值记录在实验报告中：**两个版本的执行时间之比是多少？**

### 一个值得注意的点

在以算法分析为主的课程中，衡量程序效率的标准是**操作次数**。这一判据在计算机发展的早期是充分的：当时访问内存与执行运算的速度处于同一量级。

当前的情况已完全不同。处理器的运算速度在数十年间提升了数千倍，而主存的**访问延迟**改善不足十倍，二者的差距形成了所谓的**存储墙**（memory wall）：

> 现代处理器的运算能力已远超存储系统的数据供给能力。**决定程序执行时间的，往往不是运算的次数，而是访问内存的方式。**

本实验的任务，是将"访问内存的方式"这一定性表述分解为三个可测量的量：**搬运的单位**（缓存行）、**容纳的容量**（各级缓存）、**取回的时间**（访问延迟）。


## 2. 存储层次

处理器与主存之间并非直接相连，其间设置了若干级**缓存**（cache）。设置缓存的原因只有一个：主存的访问延迟远高于处理器的运算速度，处理器无法承受相应的等待。

<!--
| 层级 | 典型容量 | 典型延迟 | 说明 |
|---|---|---|---|
| 寄存器 | 数十至数百字节 | 0 个周期 | 指令直接操作的对象，由编译器分配 |
| L1 数据缓存 | 32–64 KiB | 约 4 个周期 | 每个核心私有，速度最快、容量最小 |
| L2 缓存 | 数百 KiB 至 1 MiB | 十余个周期 | 每个核心私有 |
| L3 缓存 | 数 MiB 至数十 MiB | 数十个周期 | **由多个核心共享**，第四、五章的多线程实验将涉及这一特性 |
| 主存 (DRAM) | 数 GiB 至数百 GiB | **上百个周期** | 容量最大、延迟最高 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">层级</th>
      <th style="text-align: left;">典型容量</th>
      <th style="text-align: left;">典型延迟</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">寄存器</td>
      <td style="text-align: left;">数十至数百字节</td>
      <td style="text-align: left;">0 个周期</td>
      <td style="text-align: left;">指令直接操作的对象，由编译器分配</td>
    </tr>
    <tr>
      <td style="text-align: left;">L1 数据缓存</td>
      <td style="text-align: left;">32–64 KiB</td>
      <td style="text-align: left;">约 4 个周期</td>
      <td style="text-align: left;">每个核心私有，速度最快、容量最小</td>
    </tr>
    <tr>
      <td style="text-align: left;">L2 缓存</td>
      <td style="text-align: left;">数百 KiB 至 1 MiB</td>
      <td style="text-align: left;">十余个周期</td>
      <td style="text-align: left;">每个核心私有</td>
    </tr>
    <tr>
      <td style="text-align: left;">L3 缓存</td>
      <td style="text-align: left;">数 MiB 至数十 MiB</td>
      <td style="text-align: left;">数十个周期</td>
      <td style="text-align: left;"><strong>由多个核心共享</strong>，第四、五章的多线程实验将涉及这一特性</td>
    </tr>
    <tr>
      <td style="text-align: left;">主存 (DRAM)</td>
      <td style="text-align: left;">数 GiB 至数百 GiB</td>
      <td style="text-align: left;"><strong>上百个周期</strong></td>
      <td style="text-align: left;">容量最大、延迟最高</td>
    </tr>
  </tbody>
</table>


> 上表所列为**数量级**，具体数值随处理器型号而异。本实验的指针追逐实验将测定本机的实际数值。

### 2.1 缓存的有效性依据：局部性原理

缓存的容量较主存小若干个数量级，其之所以有效，依据的是程序访问内存时普遍表现出的两条规律：

<!--
| 规律 | 含义 | 典型来源 |
|---|---|---|
| **时间局部性** | 刚被访问的数据，在短期内很可能再次被访问 | 循环变量、累加器、被反复调用的函数 |
| **空间局部性** | 刚被访问的数据，其**相邻**的数据在短期内很可能被访问 | 顺序遍历数组、逐字段读取结构体 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">规律</th>
      <th style="text-align: left;">含义</th>
      <th style="text-align: left;">典型来源</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>时间局部性</strong></td>
      <td style="text-align: left;">刚被访问的数据，在短期内很可能再次被访问</td>
      <td style="text-align: left;">循环变量、累加器、被反复调用的函数</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>空间局部性</strong></td>
      <td style="text-align: left;">刚被访问的数据，其<strong>相邻</strong>的数据在短期内很可能被访问</td>
      <td style="text-align: left;">顺序遍历数组、逐字段读取结构体</td>
    </tr>
  </tbody>
</table>


### 2.2 缓存行：数据搬运的最小单位

利用空间局部性的具体机制是**缓存行**：

> 处理器**不会**只取回所请求的那 4 个字节，而是取回包含该地址的、按地址对齐的一整块数据，其大小通常为 **64 字节**。这一整块称为一条缓存行，它是缓存与主存之间搬运数据的最小单位。

由此可得两条推论，本实验将逐一予以验证：

1. **顺序访问的边际代价极低**。取回一条 64 字节的缓存行之后，随后的 15 个 `int` 已经位于缓存之中，仅第 1 次访问付出了搬运代价；
2. **跨越式访问的代价高昂**。若每次访问跨越 64 字节，则每次访问都需搬运一整条缓存行，而其中仅有 4 个字节被实际使用，**93.75% 的搬运量被浪费**。

第 1 节中的版本 B 即属于第二种情形：列优先遍历时，相邻两次访问在内存中相距 $N \times 4$ 字节。

### 2.3 硬件预取器

处理器还具备另一项机制：**硬件预取器**（prefetcher）。它监视访存的地址序列，一旦识别出规律（例如"每次前进 64 字节"），便在程序发出请求之前提前取回后续的缓存行。

预取器的能力边界十分重要，本实验的三个部分恰好将其完整勾勒：

- 它**能够隐藏访存延迟**——顺序扫描时数据在被需要之前即已就位，处理器无需等待；
- 它**不能减少数据搬运量**——需要搬运的字节数不会因此减少，因此带宽受限的程序不会因预取而加速（访存步长实验）；
- 它**无法预测**"下一地址由上一次读取结果决定"这一类访问模式，在该模式下完全失效（指针追逐实验正是利用这一点测定纯延迟）。


## 3. 环境准备

本实验仅依赖标准 C 库与 `matplotlib`，不需要 NEON、OpenMP 或线程库。

> ⚠️ **必须在真实硬件上运行。** 在 QEMU 等指令级模拟器中，全部指令由软件翻译执行，缓存层次被完全绕过，本实验所测得的时间数据不具备物理意义。


In [ ]:
import platform, subprocess, shutil, sys, os, glob

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
print("逻辑核心:", os.cpu_count())


def read_cache_info():
    """读取本机的存储层次参数。

    优先读取 /sys/devices/system/cpu/cpu0/cache/，该目录由内核依据处理器
    自身的标识寄存器填写，在各架构上均可用。os.sysconf 的相应键在部分平台
    （例如 aarch64）未实现，会返回 0，因此仅作为回退手段。
    """
    hw = {"缓存行": 0, "L1d": 0, "L2": 0, "L3": 0}

    def _val(path):
        try:
            with open(path) as f:
                t = f.read().strip()
        except OSError:
            return 0
        mul = 1
        if t.endswith("K"):
            mul, t = 1024, t[:-1]
        elif t.endswith("M"):
            mul, t = 1024 * 1024, t[:-1]
        try:
            return int(t) * mul
        except ValueError:
            return 0

    for d in sorted(glob.glob("/sys/devices/system/cpu/cpu0/cache/index*")):
        level = _val(f"{d}/level")
        size = _val(f"{d}/size")
        line = _val(f"{d}/coherency_line_size")
        try:
            with open(f"{d}/type") as f:
                ctype = f.read().strip()
        except OSError:
            ctype = ""
        if line and not hw["缓存行"]:
            hw["缓存行"] = line
        if level == 1 and ctype == "Data":
            hw["L1d"] = size
        elif level == 2:
            hw["L2"] = size
        elif level == 3:
            hw["L3"] = size

    for key, label in [("SC_LEVEL1_DCACHE_LINESIZE", "缓存行"),
                       ("SC_LEVEL1_DCACHE_SIZE", "L1d"),
                       ("SC_LEVEL2_CACHE_SIZE", "L2"),
                       ("SC_LEVEL3_CACHE_SIZE", "L3")]:
        if not hw[label]:
            try:
                hw[label] = os.sysconf(key)
            except (ValueError, OSError):
                pass
    return hw


# 这四个数字是本实验的对照标准：三个实验将分别独立地把它们测定一遍
HW = read_cache_info()
print("\n内核发布的存储层次参数：")
print(f"  缓存行 : {HW['缓存行']} 字节" if HW["缓存行"] else "  缓存行 : 未发布")
for lv in ("L1d", "L2", "L3"):
    v = HW[lv]
    print(f"  {lv:<6} : {v // 1024} KiB" if v else f"  {lv:<6} : 未发布")

if CC is None:
    print("\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。")
elif HW["缓存行"] == 0:
    print("\n⚠️  未能读取缓存参数，实验仍可进行，但无法与内核发布值相互核对。")
else:
    print("\n✅ 环境就绪，可以开始实验。")

In [ ]:
import subprocess, re, shutil


def compile_c(src, out):
    """编译一个 C 源文件，返回可执行文件名；失败时打印错误。
    本章不需要 -fopenmp 或 -pthread，仅使用 -O3。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC {src} -o {out} -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print("编译告警：\n", r.stderr)
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run([f"./{out}"] + [str(a) for a in args],
                       capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def _cells(line):
    return [c.strip() for c in line.strip().strip("|").split("|")]


def _isrow(line):
    s = line.strip()
    return s.startswith("|") and not set(s) <= set("-|: ")


def parse_pair(text):
    """遍历顺序实验：| 矩阵 | 字节数 | 行优先 | 列优先 | 比值 | 校验 |"""
    rows = []
    for line in text.splitlines():
        if not _isrow(line):
            continue
        c = _cells(line)
        if len(c) < 6 or "x" not in c[0]:
            continue
        try:
            rows.append({"name": c[0], "bytes": c[1], "row": float(c[2]),
                         "col": float(c[3]),
                         "ratio": float(re.search(r"[\d.]+", c[4]).group()),
                         "check": c[5]})
        except (ValueError, AttributeError):
            continue
    return rows


def parse_stride(text):
    """访存步长实验：| 步长 | 读取次数 | 单遍耗时 | ns/次 | 相对首行 | 校验 |"""
    rows = []
    for line in text.splitlines():
        if not _isrow(line):
            continue
        c = _cells(line)
        if len(c) < 6 or not c[0].isdigit():
            continue
        rows.append({"stride": int(c[0]), "reads": int(c[1]),
                     "sweep": float(c[2]), "ns": float(c[3]),
                     "rel": float(c[4].rstrip("%")), "check": c[5]})
    return rows


def parse_latency(text):
    """指针追逐实验：| 规模 | 延迟 | 相对最小规模 | 所在层级 | 校验 |"""
    rows = []
    for line in text.splitlines():
        if not _isrow(line):
            continue
        c = _cells(line)
        if len(c) < 5:
            continue
        m = re.match(r"([\d.]+)\s*(KiB|MiB)$", c[0])
        if not m:
            continue
        kib = float(m.group(1)) * (1 if m.group(2) == "KiB" else 1024)
        try:
            rows.append({"label": c[0], "kib": kib, "ns": float(c[1]),
                         "level": c[3], "check": c[4]})
        except ValueError:
            continue
    return rows


def parse_table(text):
    """通用：| 方法 | 耗时 | 加速比 | 校验 |，用于分块转置实验。"""
    rows = []
    for line in text.splitlines():
        if not _isrow(line):
            continue
        c = _cells(line)
        if len(c) < 3 or c[0].lower() in ("method", "方法"):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", c[1])
        ms = re.search(r"[-+]?\d*\.?\d+", c[2])
        if not mt:
            continue
        rows.append({"method": c[0], "time": float(mt.group()),
                     "speedup": float(ms.group()) if ms else None,
                     "check": c[3] if len(c) > 3 else ""})
    return rows

In [ ]:
import matplotlib.pyplot as plt


def plot_pair(rows, title=""):
    """遍历顺序实验：两种次序的耗时对照，附比值曲线。"""
    if not rows:
        print("未解析到数据。")
        return
    names = [r["name"] for r in rows]
    x = range(len(rows))
    w = 0.38
    fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
    ax1.bar([i - w / 2 for i in x], [r["row"] for r in rows], w,
            color="#295E96", label="Row-major")
    ax1.bar([i + w / 2 for i in x], [r["col"] for r in rows], w,
            color="#C7000B", label="Column-major")
    ax1.set_yscale("log")
    ax1.set_ylabel("Time per pass (ms, log scale)")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(names, rotation=15, ha="right")
    ax1.legend(loc="upper left", fontsize=9)
    ax2 = ax1.twinx()
    ax2.plot(list(x), [r["ratio"] for r in rows], "o--", c="#333333", lw=1.5,
             label="Col / Row")
    ax2.set_ylabel("Slowdown factor")
    ax2.set_ylim(0, max(r["ratio"] for r in rows) * 1.3)
    for i, r in enumerate(rows):
        ax2.text(i, r["ratio"], f'{r["ratio"]:.1f}x', ha="center",
                 va="bottom", fontsize=10)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_stride(rows, line_size=0, title=""):
    """访存步长实验：左轴单次读取代价，右轴整遍扫描耗时。"""
    if not rows:
        print("未解析到数据。")
        return
    s = [r["stride"] for r in rows]
    fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
    ax1.plot(s, [r["ns"] for r in rows], "o-", c="#C7000B", lw=2,
             label="Cost of ONE read (ns)")
    ax1.set_xscale("log", base=2)
    ax1.set_xticks(s)
    ax1.set_xticklabels([str(v) for v in s])
    ax1.set_xlabel("Stride (bytes, log scale)")
    ax1.set_ylabel("ns per read")
    if line_size:
        ax1.axvline(line_size, ls="--", c="gray", lw=1.2)
        ax1.text(line_size, ax1.get_ylim()[1] * 0.95,
                 f" cache line = {line_size} B", fontsize=9, va="top",
                 color="gray")
    ax2 = ax1.twinx()
    ax2.plot(s, [r["sweep"] for r in rows], "s--", c="#295E96", lw=1.5,
             label="Time for a full sweep (ms)")
    ax2.set_ylabel("Time per full sweep (ms)")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_latency(rows, hw=None, title=""):
    """指针追逐实验：延迟随工作集变化，并标出内核发布的各级缓存容量。"""
    if not rows:
        print("未解析到数据。")
        return
    x = [r["kib"] for r in rows]
    y = [r["ns"] for r in rows]
    plt.figure(figsize=(8.5, 4.5))
    plt.plot(x, y, "o-", c="#C7000B", lw=2)
    plt.xscale("log", base=2)
    plt.yscale("log")
    plt.xlabel("Working set (KiB, log scale)")
    plt.ylabel("Latency per access (ns, log scale)")
    if hw:
        for lv, color in (("L1d", "#295E96"), ("L2", "#2E8B57"),
                          ("L3", "#8A6D1F")):
            v = hw.get(lv, 0)
            if v:
                plt.axvline(v / 1024, ls="--", c=color, lw=1.2)
                plt.text(v / 1024, max(y), f" {lv}", color=color, fontsize=9,
                         va="top")
    plt.grid(True, which="both", ls=":", lw=0.5, alpha=0.6)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(f'{"Size":>10} {"Latency(ns)":>12} {"vs min":>8}  Level')
    base = y[0]
    for r in rows:
        print(f'{r["label"]:>10} {r["ns"]:>12.2f} {r["ns"]/base:>7.1f}x  {r["level"]}')


def plot_speedup(rows, title=""):
    """未通过校验的版本不进入图表。

    在本实验中校验失败意味着该版本没有写出任何结果，其耗时接近 0、
    加速比是一个无意义的巨大数值，绘入图中将使纵轴失去可读性。
    正确性优先于性能，在此处体现为一条具体的作图规则。"""
    rows = [r for r in rows if r["speedup"] is not None]
    bad = [r["method"] for r in rows if r["check"].upper() == "FAIL"]
    if bad:
        print("⚠️ 以下版本未通过校验，不参与性能比较：" + "、".join(bad))
    rows = [r for r in rows if r["check"].upper() != "FAIL"]
    if not rows:
        print("没有通过校验的版本可供绘图。请先补全分块转置实验的 TODO。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    colors = ["#295E96"] * len(sp)
    colors[sp.index(max(sp))] = "#C7000B"
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(b.get_x() + b.get_width() / 2, s, f"{s:.2f}x", ha="center",
                 va="bottom", fontsize=10)
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 创建源代码目录
!mkdir -p src_arch

## 4. 遍历顺序实验

本节回到第 1 节提出的问题。程序对同一矩阵求和两遍，二者唯一的差别是两层循环的次序；矩阵规模自 256×256（256 KiB）逐步增大至 4096×4096（64 MiB）。

### 4.1 使实验成为受控对照的两项设计

该实验成立的前提，是两个版本之间**仅有访存次序一处差异**。为此代码作了两项处理：

<!--
| 设计 | 原因 |
|---|---|
| 数据采用 `int32`，累加器采用 `int64` | 整数加法满足结合律，两个版本必须给出**逐位相同**的结果，因而可直接以 `==` 校验，无需设定容差。若采用浮点数，求和次序不同将产生微小差异，反而无法区分该差异属于正常舍入还是计算错误 |
| 两个核函数均关闭自动向量化<br>（`__attribute__((optimize("no-tree-vectorize")))`） | 行优先循环为连续访问，编译器会自动将其向量化；列优先循环不连续，无法向量化。若不予关闭，所测得的比值中将混入 SIMD 的贡献，**不再是纯粹的访存差异** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">设计</th>
      <th style="text-align: left;">原因</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">数据采用 <code>int32</code>，累加器采用 <code>int64</code></td>
      <td style="text-align: left;">整数加法满足结合律，两个版本必须给出<strong>逐位相同</strong>的结果，因而可直接以 <code>==</code> 校验，无需设定容差。若采用浮点数，求和次序不同将产生微小差异，反而无法区分该差异属于正常舍入还是计算错误</td>
    </tr>
    <tr>
      <td style="text-align: left;">两个核函数均关闭自动向量化<br>（<code>__attribute__((optimize("no-tree-vectorize")))</code>）</td>
      <td style="text-align: left;">行优先循环为连续访问，编译器会自动将其向量化；列优先循环不连续，无法向量化。若不予关闭，所测得的比值中将混入 SIMD 的贡献，<strong>不再是纯粹的访存差异</strong></td>
    </tr>
  </tbody>
</table>


In [ ]:
%%writefile src_arch/traverse_order.c
/* ==========================================================================
 * traverse_order.c -- the Traversal Order Experiment.
 *
 * A matrix of N x N 32-bit integers is summed twice:
 *   row-major     for (i) for (j)  sum += a[i * N + j];      // adjacent
 *   column-major  for (j) for (i)  sum += a[i * N + j];      // N apart
 *
 * Exactly the same elements are read, exactly the same number of times, and
 * exactly the same additions are performed. The ONLY difference is the order
 * in which memory is touched.
 *
 * Two deliberate choices keep that statement true:
 *   - integer data with a 64-bit accumulator, so both versions must produce
 *     the SAME sum bit for bit and can be compared with ==, no tolerance;
 *   - auto-vectorization is switched off, so the difference cannot come from
 *     one loop being vectorized and the other not.
 * ========================================================================== */
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// Roughly how many element reads each measurement should perform. Small
// matrices are traversed many times, large ones only once or twice, so that
// every row of the table takes a comparable amount of time.
#define TARGET_ACCESSES 33554432L   // 32 Mi

double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// ---------------------------------------------------------
// The two kernels. no-tree-vectorize is what makes this a controlled
// comparison: without it the row-major loop would also gain SIMD, and the
// measured ratio would mix two effects.
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
int64_t sum_row_major(const int32_t *a, long n) {
  int64_t sum = 0;
  for (long i = 0; i < n; i++)
    for (long j = 0; j < n; j++) sum += a[i * n + j];
  return sum;
}

#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
int64_t sum_col_major(const int32_t *a, long n) {
  int64_t sum = 0;
  for (long j = 0; j < n; j++)
    for (long i = 0; i < n; i++) sum += a[i * n + j];
  return sum;
}

int main(int argc, char **argv) {
  // Matrix orders to test. Each one is a power of two, so the stride between
  // two vertically adjacent elements is a power of two as well -- which is
  // itself part of the story (see the discussion of cache set conflicts).
  long sizes[] = {256, 512, 1024, 2048, 4096};
  int nsizes = (int)(sizeof(sizes) / sizeof(sizes[0]));
  if (argc > 1) {
    sizes[0] = atol(argv[1]);
    nsizes = 1;
  }

  printf("============================================================\n");
  printf(" Traversal Order Experiment: row-major vs column-major\n");
  printf(" Same elements, same additions, only the order differs.\n");
  printf(" Auto-vectorization is disabled in both kernels.\n");
  printf("============================================================\n");

  printf("\n---------------------------------------------------------------------\n");
  printf("| %-11s | %9s | %9s | %9s | %7s | %5s |\n", "Matrix", "Bytes",
         "Row (ms)", "Col (ms)", "Col/Row", "Check");
  printf("|-------------|-----------|-----------|-----------|---------|-------|\n");

  for (int s = 0; s < nsizes; s++) {
    long n = sizes[s];
    long nelem = n * n;
    size_t bytes = (size_t)nelem * sizeof(int32_t);

    int32_t *a = (int32_t *)malloc(bytes);
    if (!a) {
      printf("Alloc failed for n = %ld\n", n);
      return 1;
    }
    // Values are small and repeat, so the 64-bit sum cannot overflow.
    for (long i = 0; i < nelem; i++) a[i] = (int32_t)(i % 7);

    int reps = (int)(TARGET_ACCESSES / nelem);
    if (reps < 1) reps = 1;

    // Warm-up: also makes sure every page has actually been faulted in, so
    // that page faults are not charged to the first measured version.
    volatile int64_t warm = sum_row_major(a, n);
    (void)warm;

    // a[0] is rewritten before every repetition. Without it the compiler is
    // entitled to notice that the function is pure and its argument never
    // changes, run it ONCE and reuse the result -- the timing loop would then
    // measure nothing at all. The same mutation is applied to both versions,
    // so the two accumulated totals must still be identical.
    int64_t acc_r = 0, acc_c = 0;
    double t0 = get_time_ms();
    for (int r = 0; r < reps; r++) {
      a[0] = (int32_t)r;
      acc_r += sum_row_major(a, n);
    }
    double t_row = (get_time_ms() - t0) / reps;

    t0 = get_time_ms();
    for (int r = 0; r < reps; r++) {
      a[0] = (int32_t)r;
      acc_c += sum_col_major(a, n);
    }
    double t_col = (get_time_ms() - t0) / reps;

    // Both totals are built from the same integers in the same repetitions,
    // so they must be identical. Anything else means the benchmark is broken.
    const char *chk = (acc_r == acc_c) ? "PASS" : "FAIL";

    char label[32], bytelab[24];
    snprintf(label, sizeof(label), "%ldx%ld", n, n);
    if (bytes < 1024 * 1024)
      snprintf(bytelab, sizeof(bytelab), "%zu KiB", bytes / 1024);
    else
      snprintf(bytelab, sizeof(bytelab), "%zu MiB", bytes / (1024 * 1024));

    printf("| %-11s | %9s | %9.3f | %9.3f | %5.2f x | %5s |\n", label, bytelab,
           t_row, t_col, t_col / t_row, chk);
    fflush(stdout);

    free(a);
  }
  printf("---------------------------------------------------------------------\n");
  printf("  Both columns read the same number of elements. A ratio above 1.0\n");
  printf("  is therefore paid entirely for the ORDER of the accesses.\n");
  return 0;
}

In [ ]:
BIN = compile_c("src_arch/traverse_order.c", "src_arch/traverse_order")
out_ord = run_bin(BIN)

In [ ]:
rows_ord = parse_pair(out_ord)
plot_pair(rows_ord, "Traversal order: same additions, two orders")

## 5. 遍历顺序实验的结果分析

请对照所得表格，注意以下两点。

**第一，比值显著大于 1。** 两个版本读取的元素完全相同、加法次数完全相同，多出的时间**全部**由访问的**次序**造成。这构成本章的第一条认识：**操作次数不足以预测执行时间**。

**第二，比值随矩阵规模增大而上升。** 这一趋势比比值本身更具信息量：

<!--
| 矩阵规模 | 列优先版本的行为 | 结果 |
|---|---|---|
| 小至可完全驻留缓存 | 首遍将矩阵读入缓存后，后续访问无论何种次序均为命中 | 两个版本差别很小 |
| 大至无法驻留缓存 | 每次跨越均需搬运一条新的缓存行；轮到下一列使用同一条行时，该行早已被逐出 | 差距迅速扩大 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">矩阵规模</th>
      <th style="text-align: left;">列优先版本的行为</th>
      <th style="text-align: left;">结果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">小至可完全驻留缓存</td>
      <td style="text-align: left;">首遍将矩阵读入缓存后，后续访问无论何种次序均为命中</td>
      <td style="text-align: left;">两个版本差别很小</td>
    </tr>
    <tr>
      <td style="text-align: left;">大至无法驻留缓存</td>
      <td style="text-align: left;">每次跨越均需搬运一条新的缓存行；轮到下一列使用同一条行时，该行早已被逐出</td>
      <td style="text-align: left;">差距迅速扩大</td>
    </tr>
  </tbody>
</table>


换言之：**列优先的问题不在于"慢"，而在于"浪费"**。它每搬运 64 字节仅使用其中 4 字节，其余 60 字节在被使用之前即已被逐出。

> **📌 关于比值的上升趋势**
>
> 比值在工作集超出末级缓存之后会趋于**饱和**，此后各行的数值可能不再严格单调，出现小幅回落属于正常现象。原因是两个版本此时受制于不同的瓶颈：行优先版本受限于内存带宽，列优先版本除带宽外还受地址翻译（TLB）覆盖范围的影响。**应关注的是趋势与量级，而非个别数据点的排序。**

> **🤔 请先思考，访存步长实验将给出验证**
>
> 若每条缓存行为 64 字节、每个 `int` 为 4 字节，则列优先的浪费比例为 15/16。据此推算，两个版本的耗时之比"应当"是多少？表格中最大规模一行的实测值与该推算值相比是偏大还是偏小？其原因是什么？


## 6. 访存步长实验

遍历顺序实验提出了现象，但"缓存行为 64 字节"目前仍是一个未经本机验证的说法。本节将其测定出来。

实验设计如下：**取一个远大于末级缓存的数组，自头至尾扫描一遍，每次仅改变步长。**

```c
for (i = 0; i < n; i += stride) sum += a[i];
```

步长每增大一倍，读取的元素数即减半。**若"操作次数决定执行时间"成立，耗时也应当减半。** 请再次做出预测，然后对照实测结果。

### 6.1 一处关键的实现处理

核函数被手工展开为四路，使用四个独立的累加器：

```c
for (; i < lim; i += 4 * stride) {
  s0 += a[i];            s1 += a[i + stride];
  s2 += a[i + 2*stride]; s3 += a[i + 3*stride];
}
```

原因在于：步长为 4 字节时循环需执行数千万次，若仅使用一个累加器，**循环自身**（加法的依赖链与循环控制）将成为瓶颈，表格所反映的便是循环的代价而非访存的代价。四个累加器使处理器在各行中均领先于内存，从而使所有行受制于同一因素，比较方能成立。

> 这一处理本身亦构成一个知识点：**指令级并行**。打破依赖链可使处理器同时处理多条指令，这正是第三章 GEMM 优化中多累加器技巧的由来。

即便如此，**步长为 4 字节的一行仍可能受限于处理器每周期可发射的访存指令数**，而非受限于内存。程序会自动检测这一情形并在表格后予以提示；出现该提示时，应自步长 8 字节一行起读表。


In [ ]:
%%writefile src_arch/stride_scan.c
/* ==========================================================================
 * stride_scan.c -- the Stride Access Experiment.
 *
 * One array is swept from beginning to end; the only thing that changes from
 * row to row is the STRIDE, i.e. how many bytes are skipped between two
 * consecutive reads. Doubling the stride halves the number of reads.
 *
 * Intuition says the sweep should then take half the time. It does not --
 * not until the stride becomes large enough that two consecutive reads no
 * longer land in the same unit of transfer. Up to that point the memory
 * system moves exactly the same bytes, and moving bytes is what the time is
 * actually spent on.
 *
 * The program therefore watches the cost of ONE read as the stride grows.
 * It rises while several reads still share one transfer, and stops rising as
 * soon as every read needs its own. The stride at which it stops rising is
 * the effective transfer granularity, which the program reports and compares
 * against the cache line size published by the kernel.
 * ========================================================================== */
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>

#define TARGET_READS (8L * 1024 * 1024)   // per table row, roughly
#define MAX_ROWS 16

// ---------------------------------------------------------
// Cache parameters of the machine, read from sysfs.
//
// sysconf(_SC_LEVEL1_DCACHE_SIZE) and its relatives are not implemented on
// every platform; on aarch64 they commonly return 0. The kernel always
// publishes the same information under /sys/devices/system/cpu/cpu0/cache/,
// having obtained it from the processor's own identification registers, so
// that is read first and sysconf is kept only as a fallback.
// ---------------------------------------------------------
typedef struct {
  long line, l1d, l2, l3;
} cache_info_t;

static long read_size_file(const char *path) {
  FILE *f = fopen(path, "r");
  if (!f) return 0;
  char buf[64] = {0};
  if (!fgets(buf, sizeof(buf), f)) {
    fclose(f);
    return 0;
  }
  fclose(f);
  long v = atol(buf);                 // the kernel writes e.g. "512K" or "64"
  for (const char *p = buf; *p; p++) {
    if (*p == 'K') v *= 1024;
    else if (*p == 'M') v *= 1024 * 1024;
  }
  return v;
}

static cache_info_t read_cache_info(void) {
  cache_info_t c = {0, 0, 0, 0};
  char path[192];
  for (int i = 0; i < 10; i++) {
    char type[32] = {0};
    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/level", i);
    long level = read_size_file(path);
    if (!level) continue;

    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/type", i);
    FILE *f = fopen(path, "r");
    if (f) {
      if (!fgets(type, sizeof(type), f)) type[0] = '\0';
      fclose(f);
    }
    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/size", i);
    long size = read_size_file(path);
    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/coherency_line_size",
             i);
    long line = read_size_file(path);

    if (line && !c.line) c.line = line;
    if (level == 1 && type[0] == 'D') c.l1d = size;
    else if (level == 2) c.l2 = size;
    else if (level == 3) c.l3 = size;
  }
  if (!c.line) c.line = sysconf(_SC_LEVEL1_DCACHE_LINESIZE);
  if (!c.l1d) c.l1d = sysconf(_SC_LEVEL1_DCACHE_SIZE);
  if (!c.l2) c.l2 = sysconf(_SC_LEVEL2_CACHE_SIZE);
  if (!c.l3) c.l3 = sysconf(_SC_LEVEL3_CACHE_SIZE);
  return c;
}

double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// The sweep, unrolled four times with four independent accumulators.
//
// The unrolling matters for the experiment, not for the memory system: with a
// single accumulator and a stride of 4 bytes the loop itself becomes the
// bottleneck, and the table would show the cost of the LOOP instead of the
// cost of the memory traffic. Four accumulators keep the processor ahead of
// memory in every row, so all rows are limited by the same thing.
//
// Vectorization is disabled so that the stride remains the only variable.
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
int64_t sweep(const int32_t *a, long nelem, long stride) {
  int64_t s0 = 0, s1 = 0, s2 = 0, s3 = 0;
  long i = 0, lim = nelem - 3 * stride;
  for (; i < lim; i += 4 * stride) {
    s0 += a[i];
    s1 += a[i + stride];
    s2 += a[i + 2 * stride];
    s3 += a[i + 3 * stride];
  }
  for (; i < nelem; i += stride) s0 += a[i];
  return s0 + s1 + s2 + s3;
}

int main(int argc, char **argv) {
  // The array must be clearly larger than the last-level cache, otherwise it
  // would simply sit in cache and there would be no transfers to count.
  long mib = (argc > 1) ? atol(argv[1]) : 256;
  if (mib < 16) mib = 16;

  size_t bytes = (size_t)mib * 1024 * 1024;
  long nelem = (long)(bytes / sizeof(int32_t));

  int32_t *a = (int32_t *)malloc(bytes);
  if (!a) {
    printf("Alloc failed -- try a smaller size, e.g. %s 64\n", argv[0]);
    return 1;
  }
  // Every element is 1, so the sum of one sweep must equal the number of
  // elements actually read. That gives the benchmark a real self-check.
  for (long i = 0; i < nelem; i++) a[i] = 1;

  cache_info_t hw = read_cache_info();

  printf("============================================================\n");
  printf(" Stride Access Experiment: one array, growing strides\n");
  printf(" Array : %ld MiB   (must exceed the last-level cache)\n", mib);
  printf(" Every row reads HALF as many elements as the row above it.\n");
  printf("============================================================\n");

  printf("\n---------------------------------------------------------------------------\n");
  printf("| %10s | %10s | %10s | %10s | %8s | %5s |\n", "Stride (B)", "Reads",
         "Sweep (ms)", "ns / read", "vs 4 B", "Check");
  printf("|------------|------------|------------|------------|----------|-------|\n");

  double t_first = 0.0, t_second = 0.0;
  double ns_of[MAX_ROWS];
  long stride_of[MAX_ROWS];
  int nrow = 0;

  for (long stride_b = 4; stride_b <= 1024; stride_b *= 2) {
    long stride_e = stride_b / (long)sizeof(int32_t);
    long reads = (nelem + stride_e - 1) / stride_e;

    int reps = (int)(TARGET_READS / reads);
    if (reps < 1) reps = 1;
    if (reps > 100) reps = 100;

    volatile int64_t warm = sweep(a, nelem, stride_e);   // fault the pages in
    (void)warm;

    int64_t acc = 0;
    double t0 = get_time_ms();
    for (int r = 0; r < reps; r++) {
      a[0] = 1;                       // keeps the call from being hoisted out
      acc += sweep(a, nelem, stride_e);
    }
    double t_sweep = (get_time_ms() - t0) / reps;

    const char *chk = (acc == (int64_t)reads * reps) ? "PASS" : "FAIL";

    double ns = t_sweep * 1e6 / (double)reads;
    if (stride_b == 4) t_first = t_sweep;
    if (stride_b == 8) t_second = t_sweep;
    if (nrow < MAX_ROWS) {
      ns_of[nrow] = ns;
      stride_of[nrow] = stride_b;
      nrow++;
    }

    printf("| %10ld | %10ld | %10.3f | %10.2f | %7.1f%% | %5s |\n", stride_b,
           reads, t_sweep, ns, 100.0 * t_sweep / t_first, chk);
    fflush(stdout);
  }
  printf("---------------------------------------------------------------------------\n");

  // The cost of one read stops growing once every read already needs its own
  // transfer. A single flat step is not enough evidence -- the first row can
  // be limited by the loop rather than by memory -- so the plateau is only
  // accepted when TWO consecutive steps fail to grow.
  long granularity = stride_of[nrow - 1];
  for (int i = 1; i + 1 < nrow; i++) {
    if (ns_of[i] < ns_of[i - 1] * 1.25 && ns_of[i + 1] < ns_of[i] * 1.25) {
      granularity = stride_of[i - 1];
      break;
    }
  }

  printf("  Measured transfer granularity  : %ld bytes\n", granularity);
  printf("  Cache line published by kernel : %ld bytes\n", hw.line);
  if (hw.line > 0 && granularity > hw.line)
    printf("  The measured value is a MULTIPLE of the cache line. The hardware\n"
           "  prefetcher fetches neighbouring lines as well, so a sequential\n"
           "  sweep transfers more than the single line it asked for.\n");

  if (t_second > 0.0 && t_first > t_second * 1.10)
    printf("\n  Note: the 4-byte row is slower than the 8-byte row although both\n"
           "  touch the same lines. At that stride the processor has reached its\n"
           "  limit on loads issued per cycle, so this row is bound by the\n"
           "  instruction stream rather than by memory. Read the table from the\n"
           "  8-byte row onwards.\n");

  free(a);
  return 0;
}

In [ ]:
BIN = compile_c("src_arch/stride_scan.c", "src_arch/stride_scan")
# 参数：数组大小（MiB）。须远大于末级缓存；内存受限时可调小，但不应低于 L3 容量的两倍
out_str = run_bin(BIN, 256)

In [ ]:
rows_str = parse_stride(out_str)
plot_stride(rows_str, line_size=HW["缓存行"],
            title="Stride access: cost of one read as the stride grows")

## 7. 访存步长实验的结果分析

### 7.1 读取次数减少十六倍，耗时并未相应减少

请比较步长 8 字节与步长 64 字节两行：后者读取的元素数仅为前者的 1/8，而整遍扫描的耗时**远未降至 1/8**。

原因即在于缓存行：这些行所触碰的**缓存行数量完全相同**——整个数组的每一条缓存行都需被取回一次。既然搬运量相同，耗时自然接近。

> **⭐ 本实验最核心的结论**
>
> **内存的计价单位是缓存行，而非变量。** 程序请求的是 4 个字节，硬件实际搬运的是 64 个字节。

### 7.2 单次读取的代价：先上升，后持平

`ns / read` 一列是本实验真正的信号：

<!--
| 步长 | 一条缓存行被访问的次数 | 单次读取分摊的搬运量 | 曲线 |
|---|---|---|---|
| 小于缓存行 | 多次 | 缓存行大小 ÷ 访问次数，随步长成比例增大 | **上升** |
| 大于等于缓存行 | 恰好一次 | 一整条缓存行，不再变化 | **持平** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">步长</th>
      <th style="text-align: left;">一条缓存行被访问的次数</th>
      <th style="text-align: left;">单次读取分摊的搬运量</th>
      <th style="text-align: left;">曲线</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">小于缓存行</td>
      <td style="text-align: left;">多次</td>
      <td style="text-align: left;">缓存行大小 ÷ 访问次数，随步长成比例增大</td>
      <td style="text-align: left;"><strong>上升</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">大于等于缓存行</td>
      <td style="text-align: left;">恰好一次</td>
      <td style="text-align: left;">一整条缓存行，不再变化</td>
      <td style="text-align: left;"><strong>持平</strong></td>
    </tr>
  </tbody>
</table>


因此，**曲线由上升转为持平的那个步长，即为内存的有效搬运粒度**。程序会自动定位该拐点并打印，同时打印内核发布的缓存行大小，供直接核对。

判定采用的规则是：**连续两级步长均不再上升**，方认定进入平台段。仅凭单级不上升不足以判定，因为首行可能受限于指令发射而非内存。

### 7.3 测得的粒度大于 64 字节的情形

这是相当常见的结果，**并非实验失败**，而是又一项硬件事实：**硬件预取器在顺序扫描时会将相邻的若干条缓存行一并取回**。于是程序虽然只请求一条缓存行，硬件实际搬运了两条乃至四条，有效搬运粒度便成为缓存行的整数倍。

不同处理器的预取深度不同，因而该倍数在不同平台上可能是 2 倍，也可能是 4 倍。这恰好印证了第 2.3 节的论断：**预取器能够隐藏访存延迟，但不能减少数据搬运量。**

> **🔧 验证方法**：将上一单元格的参数 `256` 改为一个小于 L1 容量的数值（例如 `16` 并在程序中相应调整下限），重新运行。当数组可完全驻留缓存时，各步长的耗时将呈现何种形态？其原因是什么？


## 8. 指针追逐实验

前两个实验测定的均为**搬运量**。本节测定另一个量：**单次访问需要等待多久**。

该量的测定难度较高，因为处理器具备两项专门用于隐藏延迟的机制：

1. **乱序执行**：同时发出多个访存请求，使其等待时间相互重叠；
2. **硬件预取器**：在程序发出请求之前提前取回数据。

要测定**纯粹的**访问延迟，必须使这两项机制同时失效。所采用的方法是**指针追逐**：

> 将数组构造为一个大型**环形链表**，其链接次序**随机打乱**。每一步所要访问的地址，恰为上一步读回的值。

由此：下一个地址在当前读取完成之前**无法获知**，乱序执行无从重叠；地址序列随机，预取器无从预测。循环的执行速度即精确等于一次访存延迟。

```c
for (i = 0; i < times; i++) idx = mem[idx];   /* 一次迭代 = 一次访存延迟 */
```

### 8.1 保证测量可信的三项设计

<!--
| 设计 | 作用 |
|---|---|
| 每条缓存行仅放置**一个**链接节点 | 保证每一步均跨越至一条新的缓存行，而非在同一条行内部移动 |
| 内置**环完整性校验** | 打乱后必须形成**一个**大环。若碎裂为若干小环，程序将仅在其中一个中循环，实际工作集远小于设定值，所得延迟偏低，**且不会产生任何报错**。校验自 0 出发行走 $L$ 步，要求恰在第 $L$ 步回到起点 |
| 固定随机种子并内置随机数发生器 | 不使用 `rand()`，因其序列取决于 C 库版本。如此可保证任何机器、任何一次运行所得排列一致，结果可复现 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">设计</th>
      <th style="text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">每条缓存行仅放置<strong>一个</strong>链接节点</td>
      <td style="text-align: left;">保证每一步均跨越至一条新的缓存行，而非在同一条行内部移动</td>
    </tr>
    <tr>
      <td style="text-align: left;">内置<strong>环完整性校验</strong></td>
      <td style="text-align: left;">打乱后必须形成<strong>一个</strong>大环。若碎裂为若干小环，程序将仅在其中一个中循环，实际工作集远小于设定值，所得延迟偏低，<strong>且不会产生任何报错</strong>。校验自 0 出发行走 L 步，要求恰在第 L 步回到起点</td>
    </tr>
    <tr>
      <td style="text-align: left;">固定随机种子并内置随机数发生器</td>
      <td style="text-align: left;">不使用 <code>rand()</code>，因其序列取决于 C 库版本。如此可保证任何机器、任何一次运行所得排列一致，结果可复现</td>
    </tr>
  </tbody>
</table>


程序将扫描 4 KiB 至 128 MiB 的工作集，并在最后自动报告延迟发生跳变的位置。


In [ ]:
%%writefile src_arch/cache_latency.c
/* ==========================================================================
 * cache_latency.c -- the Pointer Chasing Experiment.
 *
 * How long does ONE memory read take? There is no single answer: it depends
 * on which level of the hierarchy the data happens to be in.
 *
 * Measuring it requires care, because a modern processor is very good at
 * hiding memory latency: it issues many reads at once and prefetches what it
 * expects to be asked for next. Both tricks are defeated here by POINTER
 * CHASING: the array is turned into one long circular linked list whose links
 * are in random order, so
 *
 *     the address of the next read is the value returned by the current one.
 *
 * Nothing can be issued in parallel and nothing can be predicted; the loop
 * runs at exactly one memory latency per iteration.
 *
 * Sweeping the size of the array then draws the whole hierarchy: the array
 * fits in L1, then only in L2, then only in L3, then not at all, and the
 * measured latency steps up at each of those boundaries.
 * ========================================================================== */
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>

// One element of every cache line is used as a link; the rest of the line is
// never touched, so that every step of the chase costs one full line fetch.
// 64 B is the line size of both x86-64 and the Kunpeng 920 -- and it is not
// assumed here, it is what the Stride Access Experiment measures
// independently.
#define LINE_BYTES 64

#define WARMUP_CHASES 200000L
#define TARGET_MS 60.0            // how long each measurement should take
#define MIN_CHASES 200000L
#define MAX_CHASES 40000000L

// Keeps the traversal loop from being optimized away.
volatile uint32_t global_sink = 0;

// ---------------------------------------------------------
// Cache parameters of the machine, read from sysfs.
//
// sysconf(_SC_LEVEL1_DCACHE_SIZE) and its relatives are not implemented on
// every platform; on aarch64 they commonly return 0. The kernel always
// publishes the same information under /sys/devices/system/cpu/cpu0/cache/,
// having obtained it from the processor's own identification registers, so
// that is read first and sysconf is kept only as a fallback.
// ---------------------------------------------------------
typedef struct {
  long line, l1d, l2, l3;
} cache_info_t;

static long read_size_file(const char *path) {
  FILE *f = fopen(path, "r");
  if (!f) return 0;
  char buf[64] = {0};
  if (!fgets(buf, sizeof(buf), f)) {
    fclose(f);
    return 0;
  }
  fclose(f);
  long v = atol(buf);                 // the kernel writes e.g. "512K" or "64"
  for (const char *p = buf; *p; p++) {
    if (*p == 'K') v *= 1024;
    else if (*p == 'M') v *= 1024 * 1024;
  }
  return v;
}

static cache_info_t read_cache_info(void) {
  cache_info_t c = {0, 0, 0, 0};
  char path[192];
  for (int i = 0; i < 10; i++) {
    char type[32] = {0};
    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/level", i);
    long level = read_size_file(path);
    if (!level) continue;

    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/type", i);
    FILE *f = fopen(path, "r");
    if (f) {
      if (!fgets(type, sizeof(type), f)) type[0] = '\0';
      fclose(f);
    }
    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/size", i);
    long size = read_size_file(path);
    snprintf(path, sizeof(path),
             "/sys/devices/system/cpu/cpu0/cache/index%d/coherency_line_size",
             i);
    long line = read_size_file(path);

    if (line && !c.line) c.line = line;
    if (level == 1 && type[0] == 'D') c.l1d = size;
    else if (level == 2) c.l2 = size;
    else if (level == 3) c.l3 = size;
  }
  if (!c.line) c.line = sysconf(_SC_LEVEL1_DCACHE_LINESIZE);
  if (!c.l1d) c.l1d = sysconf(_SC_LEVEL1_DCACHE_SIZE);
  if (!c.l2) c.l2 = sysconf(_SC_LEVEL2_CACHE_SIZE);
  if (!c.l3) c.l3 = sysconf(_SC_LEVEL3_CACHE_SIZE);
  return c;
}

double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// A tiny deterministic generator. rand() is avoided on purpose: its sequence
// depends on the C library, so the same experiment would shuffle differently
// on different machines. Here the permutation -- and therefore the result --
// is reproducible everywhere.
static uint64_t rng_state = 88172645463325252ULL;

static void rng_seed(uint64_t s) { rng_state = s ? s : 1; }

static uint64_t rng_next(void) {
  uint64_t x = rng_state;
  x ^= x << 13;
  x ^= x >> 7;
  x ^= x << 17;
  rng_state = x;
  return x;
}

// Fisher-Yates. The modulo is applied to a 64-bit value over a range far
// below 2^32, so the bias is negligible.
static void shuffle(uint32_t *v, size_t n) {
  for (size_t i = n - 1; i > 0; i--) {
    size_t j = (size_t)(rng_next() % (uint64_t)(i + 1));
    uint32_t t = v[i];
    v[i] = v[j];
    v[j] = t;
  }
}

// The chase itself, in its own function so that the timed loop contains
// nothing else.
static uint32_t chase(const uint32_t *mem, uint32_t idx, long times) {
  for (long i = 0; i < times; i++) idx = mem[idx];
  return idx;
}

// Returns the average latency in nanoseconds, and writes PASS/FAIL of the
// ring self-check into *chk.
static double measure(size_t size_bytes, const char **chk) {
  const size_t stride = LINE_BYTES / sizeof(uint32_t);   // 16 elements
  const size_t nelem = size_bytes / sizeof(uint32_t);
  const size_t nlines = size_bytes / LINE_BYTES;

  uint32_t *mem = (uint32_t *)calloc(nelem, sizeof(uint32_t));
  uint32_t *order = (uint32_t *)malloc(nlines * sizeof(uint32_t));
  if (!mem || !order) {
    printf("Allocation failed for %zu bytes\n", size_bytes);
    exit(EXIT_FAILURE);
  }

  for (size_t i = 0; i < nlines; i++) order[i] = (uint32_t)i;
  rng_seed(20260822ULL);          // same permutation on every machine, every run
  shuffle(order, nlines);

  // Link the lines into ONE circular list following the shuffled order.
  for (size_t i = 0; i + 1 < nlines; i++)
    mem[order[i] * stride] = (uint32_t)(order[i + 1] * stride);
  mem[order[nlines - 1] * stride] = (uint32_t)(order[0] * stride);

  // Self-check: starting from line 0, exactly nlines steps must be needed to
  // come back to it. A shorter return would mean the shuffle produced several
  // small rings instead of one big one, and the measurement would then only
  // walk a fraction of the array -- silently reporting the wrong level.
  uint32_t p = mem[0];
  size_t steps = 1;
  while (p != 0 && steps <= nlines) {
    p = mem[p];
    steps++;
  }
  *chk = (p == 0 && steps == nlines) ? "PASS" : "FAIL";

  // Warm-up: brings the data in (as far as it fits) and faults in every page.
  uint32_t idx = chase(mem, 0, WARMUP_CHASES);

  // Calibrate, so that a fast level is not measured for a whole second and a
  // slow one is not measured for a millisecond.
  double t0 = get_time_ms();
  idx = chase(mem, idx, MIN_CHASES);
  double probe_ms = get_time_ms() - t0;
  double ns_guess = probe_ms * 1e6 / (double)MIN_CHASES;
  long chases = (long)(TARGET_MS * 1e6 / (ns_guess > 0.1 ? ns_guess : 0.1));
  if (chases < MIN_CHASES) chases = MIN_CHASES;
  if (chases > MAX_CHASES) chases = MAX_CHASES;

  t0 = get_time_ms();
  idx = chase(mem, idx, chases);
  double total_ms = get_time_ms() - t0;

  global_sink = idx;              // the result must be used, or the loop dies

  free(mem);
  free(order);
  return total_ms * 1e6 / (double)chases;
}

static void human(char *buf, size_t buflen, size_t bytes) {
  if (bytes < 1024 * 1024)
    snprintf(buf, buflen, "%zu KiB", bytes / 1024);
  else
    snprintf(buf, buflen, "%zu MiB", bytes / (1024 * 1024));
}

int main(int argc, char **argv) {
  size_t max_mib = (argc > 1) ? (size_t)atol(argv[1]) : 128;
  if (max_mib < 1) max_mib = 128;

  cache_info_t hw = read_cache_info();
  long l1 = hw.l1d, l2 = hw.l2, l3 = hw.l3;

  printf("============================================================\n");
  printf(" Pointer Chasing Experiment: the latency pyramid\n");
  printf(" Caches published by the kernel: L1d %ld KiB | L2 %ld KiB | L3 %ld KiB\n",
         l1 / 1024, l2 / 1024, l3 / 1024);
  printf(" Each read depends on the previous one, so nothing can be\n");
  printf(" overlapped and nothing can be prefetched.\n");
  printf("============================================================\n");

  printf("\n----------------------------------------------------------------\n");
  printf("| %10s | %12s | %9s | %8s | %5s |\n", "Size", "Latency (ns)",
         "vs 4 KiB", "Fits in", "Check");
  printf("|------------|--------------|-----------|----------|-------|\n");

  double first = 0.0, prev = 0.0;
  size_t prev_size = 0;
  // Boundaries the table crosses, filled in as the latency jumps
  size_t knee[8];
  double knee_jump[8];
  int nknee = 0;

  for (size_t size = 4 * 1024; size <= max_mib * 1024 * 1024; size *= 2) {
    const char *chk = "-";
    double ns = measure(size, &chk);
    if (first == 0.0) first = ns;

    if (prev > 0.0 && ns > prev * 1.50 && nknee < 8) {
      knee[nknee] = prev_size;
      knee_jump[nknee] = ns / prev;
      nknee++;
    }
    prev = ns;
    prev_size = size;

    // Where the OS says a working set of this size should still fit. The
    // table can then be read against the hardware specification directly.
    const char *level = "DRAM";
    if (l3 > 0 && (long)size <= l3) level = "L3";
    if (l2 > 0 && (long)size <= l2) level = "L2";
    if (l1 > 0 && (long)size <= l1) level = "L1d";

    char lab[24];
    human(lab, sizeof(lab), size);
    printf("| %10s | %12.2f | %7.2f x | %8s | %5s |\n", lab, ns, ns / first,
           level, chk);
    fflush(stdout);
  }
  printf("----------------------------------------------------------------\n");
  printf("  'Fits in' is where the kernel-reported cache sizes say the working\n");
  printf("  set should still fit. Compare it with where the latency jumps.\n");

  if (nknee) {
    printf("\n  Latency steps up right after these sizes:\n");
    for (int i = 0; i < nknee; i++) {
      char lab[24];
      human(lab, sizeof(lab), knee[i]);
      printf("    %10s  ->  x%.2f\n", lab, knee_jump[i]);
    }
    printf("  Each of them is a cache the working set has just outgrown. The last\n");
    printf("  step may appear earlier than the nominal L3 size, because the L3 is\n");
    printf("  shared with the other cores and because address translation (TLB)\n");
    printf("  runs out of reach at about the same point.\n");
  }
  return 0;
}

In [ ]:
BIN = compile_c("src_arch/cache_latency.c", "src_arch/cache_latency")
# 参数：最大工作集（MiB）。应明显大于本机 L3 容量，否则无法观察到最后一级台阶
out_lat = run_bin(BIN, 128)

In [ ]:
rows_lat = parse_latency(out_lat)
plot_latency(rows_lat, hw=HW, title="Pointer chasing: latency vs working set")

## 9. 三个实验的相互印证

至此，三个互不相同的实验各自独立地测定了同一套硬件参数。请将其并列核对——**这一步是本实验的落脚点**。

<!--
| 待测参数 | 测定实验 | 读取方式 | 核对对象 |
|---|---|---|---|
| **缓存行大小** | 访存步长实验 | 单次读取代价由上升转为持平的步长 | 第 3 节打印的缓存行大小 |
| **各级缓存容量** | 指针追逐实验 | 延迟曲线上各级台阶的起始位置 | 第 3 节打印的 L1d / L2 / L3 容量 |
| **各级访问延迟** | 指针追逐实验 | 各级台阶的高度 | 第 2 节表格给出的数量级 |
| **上述参数的后果** | 遍历顺序实验 | 列优先与行优先之比随规模的变化 | 由前两者予以解释 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">待测参数</th>
      <th style="text-align: left;">测定实验</th>
      <th style="text-align: left;">读取方式</th>
      <th style="text-align: left;">核对对象</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>缓存行大小</strong></td>
      <td style="text-align: left;">访存步长实验</td>
      <td style="text-align: left;">单次读取代价由上升转为持平的步长</td>
      <td style="text-align: left;">第 3 节打印的缓存行大小</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>各级缓存容量</strong></td>
      <td style="text-align: left;">指针追逐实验</td>
      <td style="text-align: left;">延迟曲线上各级台阶的起始位置</td>
      <td style="text-align: left;">第 3 节打印的 L1d / L2 / L3 容量</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>各级访问延迟</strong></td>
      <td style="text-align: left;">指针追逐实验</td>
      <td style="text-align: left;">各级台阶的高度</td>
      <td style="text-align: left;">第 2 节表格给出的数量级</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>上述参数的后果</strong></td>
      <td style="text-align: left;">遍历顺序实验</td>
      <td style="text-align: left;">列优先与行优先之比随规模的变化</td>
      <td style="text-align: left;">由前两者予以解释</td>
    </tr>
  </tbody>
</table>


### 9.1 请回答以下三个问题

1. **指针追逐实验的第一级台阶**出现在何种规模？与内核发布的 L1d 容量是否一致？若存在偏差，可能的原因是什么？（提示：链接节点本身亦占用缓存容量，页表项缓存 TLB 同样需要占用资源。）

2. **指针追逐实验的最后一级台阶**是否明显早于内核发布的 L3 容量？这在多核处理器上十分常见，原因有二：L3 由全部核心**共享**，单个程序无法独占；同时地址翻译所需的 TLB 覆盖范围也在相近规模上耗尽。此外，L2 至 L3 之间的过渡可能表现为**若干级较缓的上升**而非一次陡峭跳变，这与缓存的组相联结构及预取策略有关。

3. **回到遍历顺序实验**：取表格中最大的矩阵，用另外两个实验的结果解释其比值。列优先每访问一个 `int` 需搬运一整条缓存行，浪费率为 15/16，据此推算的比值为 16。实测值与之相比偏大还是偏小？若偏小，可能的原因是列优先版本亦部分受益于预取，或受限于其他瓶颈而未能充分暴露差距。

> **⭐ 需要掌握的三条结论**
>
> 1. **决定性能的不仅是算法的操作次数，还有数据的访问方式。** 二者对执行时间的影响可相差一个数量级，而只有前者会出现在算法复杂度分析中。
> 2. **内存以缓存行为单位计价。** 编写代码时应当提出的问题是："所搬入的这条缓存行，是否被充分使用？"
> 3. **硬件参数可以实测，不必仅依赖手册。** 三段不足百行的 C 代码即可测定本机的缓存行大小与各级缓存容量。


## 10. 🚀 分块转置实验

前三个实验均属**测量**。本节的任务是**优化**：将所得认识用于改进一段程序的性能。

对象是**矩阵转置**，它是遍历顺序实验中列优先访问的一个无法回避的实例：

```c
for (i = 0; i < n; i++)
  for (j = 0; j < n; j++)
    b[j * n + i] = a[i * n + j];
/*      ^ 按列写入 B        ^ 按行读取 A  */
```

无论如何交换两层循环，总有一个矩阵按列访问，**次序问题在此无解**。

### 10.1 分块（blocking / tiling）

解决途径不是改变元素的访问次序，而是改变**一次处理的区域大小**：

> 不再一次转置整行，而是一次转置一个 $T \times T$ 的子块。只要 $T$ 足够小，使 A 的子块与 B 的子块能够同时驻留于缓存，则每一条搬入的缓存行都将在被逐出之前**被完全使用**。

这就把空间局部性由一项观察转化为一个**可主动使用的工具**。同一思想在第三章将被用于矩阵乘法，在那里称为 **Cache 分块**。

### 10.2 待完成的任务

请补全 `transpose_blocked()` 中的两处 TODO：

- **TODO 1**：外两层循环按块推进（步长为 `t`），并计算当前块的边界。注意 `n` 未必是 `t` 的整数倍，末块规模更小，必须将边界钳制至 `n`；
- **TODO 2**：内两层循环遍历当前块内的元素，赋值语句与朴素版本完全一致。

未补全时程序可正常编译运行，但所有 `Blocked` 行均报 `FAIL`——因为没有向 `b` 写入任何内容。补全后应全部 `PASS`，且至少有一个块大小明显快于朴素版本。


In [ ]:
%%writefile src_arch/transpose_ext.c
/* ==========================================================================
 * transpose_ext_skeleton.c -- the Blocked Transpose Experiment (exercise).
 *
 * Complete the two TODOs in transpose_blocked(). Everything else is written.
 * Until they are completed the program compiles and runs, and every blocked
 * row reports FAIL, because nothing is ever written into b.
 *
 * The Traversal Order Experiment showed that a column-major traversal is
 * slow. A matrix transpose is the case where that cannot simply be avoided:
 * whichever way the loops are written, one of the two matrices is walked
 * down a column.
 *
 *     for (i) for (j)  b[j * n + i] = a[i * n + j];
 *              ^ reads a row of A          ^ writes a column of B
 *
 * The way out is not to change the order of the elements but the SIZE of the
 * region worked on: transpose one small tile at a time, small enough that the
 * tile of A and the tile of B both stay in cache while they are being used.
 * Every cache line brought in is then fully consumed before it is evicted.
 *
 * This is the same idea that Chapter 3 will apply to matrix multiplication.
 * ========================================================================== */
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 3

double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// ---------------------------------------------------------
// The straightforward transpose, and the reference result.
// ---------------------------------------------------------
void transpose_naive(const int32_t *a, int32_t *b, long n) {
  for (long i = 0; i < n; i++)
    for (long j = 0; j < n; j++) b[j * n + i] = a[i * n + j];
}

// ---------------------------------------------------------
// The blocked transpose. Same elements, same assignments, different order.
// ---------------------------------------------------------
void transpose_blocked(const int32_t *a, int32_t *b, long n, long t) {
  /* TODO 1: advance over the matrix one tile at a time.
   *   Two loops, ii and jj, both stepping by t instead of by 1.
   *   Inside them, work out where the current tile ENDS. When n is not a
   *   multiple of t the last tile of a row or column is smaller, so the end
   *   must be clamped to n -- this is what makes the code work for any n.
   *
   * TODO 2: transpose the elements of the current tile.
   *   Two more loops, i and j, running over that tile only, with the same
   *   assignment as the naive version:  b[j * n + i] = a[i * n + j];
   *
   * The four casts below only keep the compiler from warning that the
   * parameters are unused. Delete them once you have written the loops. */
  (void)a;
  (void)b;
  (void)n;
  (void)t;
}

int main(int argc, char **argv) {
  long n = (argc > 1) ? atol(argv[1]) : 4096;
  if (n < 8) n = 4096;

  long tiles[] = {8, 16, 32, 64, 128};
  int ntiles = (int)(sizeof(tiles) / sizeof(tiles[0]));

  size_t bytes = (size_t)n * (size_t)n * sizeof(int32_t);
  int32_t *a = (int32_t *)malloc(bytes);
  int32_t *ref = (int32_t *)malloc(bytes);
  int32_t *b = (int32_t *)malloc(bytes);
  if (!a || !ref || !b) {
    printf("Alloc failed\n");
    return 1;
  }
  for (long i = 0; i < n * n; i++) a[i] = (int32_t)(i * 2654435761u);

  printf("============================================================\n");
  printf(" Blocked Transpose Experiment\n");
  printf(" Matrix : %ld x %ld int32  (%.1f MiB per matrix)\n", n, n,
         (double)bytes / (1024.0 * 1024.0));
  printf(" A tile of %ld x %ld int32 occupies %ld B; two of them must fit\n",
         tiles[0], tiles[0], tiles[0] * tiles[0] * 4);
  printf(" comfortably in L1 for the blocking to pay off.\n");
  printf("============================================================\n");

  // Golden reference
  transpose_naive(a, ref, n);

  printf("\n------------------------------------------------------\n");
  printf("| %-16s | %9s | %7s | %5s |\n", "Method", "Time (ms)", "Speedup",
         "Check");
  printf("|------------------|-----------|---------|-------|\n");

  memset(b, 0, bytes);
  double t0 = get_time_ms();
  for (int r = 0; r < NTIMES; r++) transpose_naive(a, b, n);
  double t_base = (get_time_ms() - t0) / NTIMES;
  printf("| %-16s | %9.3f | %5.2f x | %5s |\n", "Naive", t_base, 1.0,
         memcmp(ref, b, bytes) == 0 ? "PASS" : "FAIL");

  double best_t = t_base;
  long best_tile = 0;
  for (int k = 0; k < ntiles; k++) {
    // Clear: otherwise a correct result left by the previous variant would
    // mask a variant that writes nothing at all.
    memset(b, 0, bytes);
    t0 = get_time_ms();
    for (int r = 0; r < NTIMES; r++) transpose_blocked(a, b, n, tiles[k]);
    double t = (get_time_ms() - t0) / NTIMES;

    int ok = (memcmp(ref, b, bytes) == 0);
    char name[32];
    snprintf(name, sizeof(name), "Blocked %ld", tiles[k]);
    printf("| %-16s | %9.3f | %5.2f x | %5s |\n", name, t, t_base / t,
           ok ? "PASS" : "FAIL");
    // Only a variant that produced the right matrix may claim to be fastest.
    if (ok && t < best_t) {
      best_t = t;
      best_tile = tiles[k];
    }
  }
  printf("------------------------------------------------------\n");
  if (best_tile)
    printf("  Best tile: %ld x %ld  (%.2f x faster than the naive version)\n",
           best_tile, best_tile, t_base / best_t);
  else
    printf("  No tile size beat the naive version on this machine.\n");

  free(a);
  free(ref);
  free(b);
  return 0;
}

In [ ]:
BIN = compile_c("src_arch/transpose_ext.c", "src_arch/transpose_ext")
# 参数：矩阵阶数。补全 TODO 之前，各 Blocked 行应报 FAIL
out_tr = run_bin(BIN, 4096)
rows_tr = parse_table(out_tr)
plot_speedup(rows_tr, "Blocked transpose: naive vs blocked")

## 11. 🔧 动手练习

1. **补全分块转置实验的两处 TODO**，直至所有行 `PASS`。随后分别以 `4096`（2 的幂）与 `4000`、`1000`（非块大小的整数倍）验证，确认边界处理正确。

2. **解释最优块大小**。表格中哪一个 $T$ 最快？将 $2 \times T \times T \times 4$ 字节（一块 A 加一块 B）与第 3 节所报告的 L1d 容量相比较，二者呈何种关系？$T$ 继续增大为何反而变慢？

3. **改变访存步长实验的数组规模**。依次以 `16`、`64`、`256`（MiB）运行，将三条 `ns / read` 曲线绘于同一图中。当数组小于 L1 容量时曲线是否仍有拐点？其原因是什么？

4. **消除遍历顺序实验的差距**。在不改变列优先循环次序的前提下，仅改变**矩阵的存储方式**，使列优先版本达到与行优先版本相当的性能。（提示：若矩阵本身即按列存储会如何？这说明性能取决于循环本身，还是取决于循环与数据布局的**匹配**？）

5. **测定结构体布局的影响**。定义一个含 16 个 `int` 字段的结构体，建立一个长数组，随后分别：(a) 遍历该数组，仅读取第一个字段；(b) 将 16 个字段拆分为 16 个独立数组，仅遍历第一个数组。两者读取的有效数据量相同，耗时相差多少？这两种布局在图形与游戏编程中分别称为 AoS 与 SoA。


## 12. 🤔 思考题

1. 遍历顺序实验中的矩阵阶数均取 2 的幂。若将 4096 改为 4097，列优先版本的耗时是否会明显变化？（提示：缓存采用**组相联**结构，地址中的若干位决定数据映射至哪一组。当列间距恰为 2 的较大幂时，同一列的元素易于全部映射至同一组而相互驱逐，此现象称为**缓存冲突**。）

2. 指针追逐实验采用随机次序链接节点。若改为顺序链接（`mem[i] = i + 16`），所测得的"延迟"将变为何值？该值是否仍是延迟？

3. 第 2.3 节指出预取器"不能减少数据搬运量"。据此判断：**带宽受限**的程序（例如向量加法 `c[i] = a[i] + b[i]`）能否从预取中获益？**延迟受限**的程序（例如遍历链表）呢？

4. 指针追逐实验的横轴止于 128 MiB。若继续增大至 8 GiB，延迟是否会再上升一级台阶？该台阶对应何种机制？（提示：除数据需要缓存外，虚拟地址至物理地址的映射同样需要缓存。）

5. 本实验测得的全部数值均属于**单个核心**。第四、五章将使多个核心同时工作。届时 L1 与 L2 为每核私有，而 L3 与内存带宽为共享资源——这将使哪些数值发生变化？哪些保持不变？

6. 回到最初的问题：若程序性能主要由访存决定，那么"该算法为 $O(n^2)$、另一算法为 $O(n \log n)$"这类分析是否仍然有效？在何种条件下它是可靠的，在何种条件下会失效？


## 13. 小结与后续

本实验以三段小程序，将第二章的抽象概念转化为本机的具体数值：

<!--
| 实验 | 测定内容 | 对应概念 |
|---|---|---|
| **遍历顺序实验** | 交换循环次序导致耗时相差一个数量级 | 空间局部性；操作次数不足以预测执行时间 |
| **访存步长实验** | 本机的缓存行大小与有效搬运粒度 | 缓存行；硬件预取器 |
| **指针追逐实验** | 本机各级缓存的容量与访问延迟 | 存储层次；时间局部性 |
| **分块转置实验** | 相同的赋值运算，改变分块方式即可获得数倍加速 | 分块——将局部性转化为可用的优化手段 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">实验</th>
      <th style="text-align: left;">测定内容</th>
      <th style="text-align: left;">对应概念</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>遍历顺序实验</strong></td>
      <td style="text-align: left;">交换循环次序导致耗时相差一个数量级</td>
      <td style="text-align: left;">空间局部性；操作次数不足以预测执行时间</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>访存步长实验</strong></td>
      <td style="text-align: left;">本机的缓存行大小与有效搬运粒度</td>
      <td style="text-align: left;">缓存行；硬件预取器</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>指针追逐实验</strong></td>
      <td style="text-align: left;">本机各级缓存的容量与访问延迟</td>
      <td style="text-align: left;">存储层次；时间局部性</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>分块转置实验</strong></td>
      <td style="text-align: left;">相同的赋值运算，改变分块方式即可获得数倍加速</td>
      <td style="text-align: left;">分块——将局部性转化为可用的优化手段</td>
    </tr>
  </tbody>
</table>


### 三条一般性结论

1. **算法的操作次数只决定性能的上限，访存模式决定了其中可实际获得的比例。**
2. **内存以缓存行为单位计价**，因此访存优化的核心问题始终是：所搬入的缓存行是否被充分使用。
3. **硬件并非黑箱。** 缓存行大小与各级缓存容量均可由数十行 C 代码测定，并与内核发布的参数相互印证。

### 与后续章节的联系

<!--
| 章节 | 关系 |
|---|---|
| 第三章 · ARM NEON SIMD | SIMD 一次处理 4 个 `float` 的前提是这 4 个数在内存中**连续**，即本实验所述的空间局部性。第三章综合实训中的 **Cache 分块**与**内存打包**，是分块转置实验中分块思想的直接延续 |
| 第四章 · Pthreads | 多个线程写入**同一条缓存行**内的不同变量时，该行将在各核心的私有缓存之间反复迁移，此即"伪共享"。理解该现象的前提正是本实验所建立的缓存行概念 |
| 第五章 · OpenMP | 多核共享 L3 与内存带宽。本实验测得的单核延迟与带宽，是判断多线程程序能否获得线性加速的基准 |
| 贯穿全课程 | 此后每遇到"未达到理论加速比"的情形，首先应当提出的问题都是：**数据位于哪一层？搬运量是多少？** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">章节</th>
      <th style="text-align: left;">关系</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">第三章 · ARM NEON SIMD</td>
      <td style="text-align: left;">SIMD 一次处理 4 个 <code>float</code> 的前提是这 4 个数在内存中<strong>连续</strong>，即本实验所述的空间局部性。第三章综合实训中的 <strong>Cache 分块</strong>与<strong>内存打包</strong>，是分块转置实验中分块思想的直接延续</td>
    </tr>
    <tr>
      <td style="text-align: left;">第四章 · Pthreads</td>
      <td style="text-align: left;">多个线程写入<strong>同一条缓存行</strong>内的不同变量时，该行将在各核心的私有缓存之间反复迁移，此即"伪共享"。理解该现象的前提正是本实验所建立的缓存行概念</td>
    </tr>
    <tr>
      <td style="text-align: left;">第五章 · OpenMP</td>
      <td style="text-align: left;">多核共享 L3 与内存带宽。本实验测得的单核延迟与带宽，是判断多线程程序能否获得线性加速的基准</td>
    </tr>
    <tr>
      <td style="text-align: left;">贯穿全课程</td>
      <td style="text-align: left;">此后每遇到"未达到理论加速比"的情形，首先应当提出的问题都是：<strong>数据位于哪一层？搬运量是多少？</strong></td>
    </tr>
  </tbody>
</table>

